# 🎬 Presentation Automator (PRA v1.0)

Este cuaderno automatiza el flujo de trabajo para la construcción de **presentaciones interactivas Reveal.js** dentro del módulo Laravel de presentaciones dinámicas. Utiliza la misma filosofía de **plan maestro + construcción progresiva por secciones** que el Asistente de Documentación IEEE, adaptada a láminas, temas Blade, CSS de clases (sin inline) y JavaScript comentado y acotado por lámina.

## 🔄 Flujo de Trabajo:
1. **Generación de Metaprompt de Plan**: A partir de un documento fuente (PDF, notebook, paper, código, resultados), genera el prompt que produce `presentation_plan.json` + los registros iniciales de clases CSS y comportamientos JS.
2. **Gestión de Plan**: Captura y guarda el plan de presentación en `presentation_plan.md`.
3. **Procesamiento del Plan**: Crea la estructura de carpetas por sesión y los registros (`class_registry.json`, `js_registry.json`) que se irán actualizando a medida que se construye cada sesión.
4. **Construcción Progresiva por Sesión**: Genera el prompt específico de cada sesión (consultando el plan + el estado actual de los registros), y un procesador directo que inyecta la respuesta de la IA (láminas Blade, adiciones a `styles.blade.php`/`scripts.blade.php`, entradas de `manifest.blade.php`) de vuelta al proyecto, actualizando los registros para la siguiente sesión.
5. **Compresión y descarga**: Empaqueta todo lo generado en un `.zip`.

**Objetivo:** Mantener coherencia visual y funcional (sin CSS inline, sin JS duplicado o mal acotado) a medida que la presentación crece sesión a sesión, sin tener que recordar manualmente qué clases o comportamientos ya existen.

## 📥 0. Plantillas del Repositorio

Este cuaderno espera encontrar dos plantillas en `templetes/`, tal como ya haces con `templete_meta_prompt.md` y `section_meta_prompt.md` para el flujo de investigación:

- `templetes/presentation_plan_meta_prompt.md`
- `templetes/slide_group_meta_prompt.md`

Si ambos archivos viven en el mismo repositorio de GitHub del que ya traes tus plantillas de investigación, usa la celda siguiente para clonarlo/actualizarlo. Si prefieres subir los archivos manualmente al entorno de Colab, puedes omitir esta celda.

In [ ]:
# @title 📦 (Opcional) Clonar / Actualizar repositorio de plantillas
# @markdown Completa la URL de tu repositorio de GitHub y ejecuta esta celda si aún no tienes la carpeta `templetes/` en este entorno.

REPO_URL = "https://github.com/tu-usuario/tu-repositorio.git"  # @param {type:"string"}

import os

if not os.path.exists('.git') and not os.path.exists('templetes'):
    print(f"⬇️  Clonando repositorio: {REPO_URL}")
    !git clone {REPO_URL} temp_repo
    if os.path.exists('temp_repo/templetes'):
        !cp -r temp_repo/templetes .
        print("✅ Carpeta 'templetes/' copiada al entorno.")
    else:
        print("⚠️  El repositorio clonado no contiene una carpeta 'templetes/'. Ajusta la ruta o sube los archivos manualmente.")
    !rm -rf temp_repo
else:
    print("ℹ️  Ya existe un repositorio o carpeta 'templetes/' en este entorno. Si quieres forzar la actualización, bórrala y vuelve a correr esta celda.")

# Verificación de las dos plantillas específicas de este flujo
for tpl in ['templetes/presentation_plan_meta_prompt.md', 'templetes/slide_group_meta_prompt.md']:
    estado = "✅ encontrada" if os.path.exists(tpl) else "❌ NO encontrada"
    print(f"{estado}: {tpl}")

## 📘 1. Generador de Metaprompt de Plan de Presentación

Captura el contenido/tema del documento fuente y el tipo de documento, y sustituye los placeholders `{{contenido_fuente}}` / `{{tipo_documento}}` en la plantilla `presentation_plan_meta_prompt.md`. El resultado es el prompt completo que debes pegar en tu IA de preferencia para obtener `presentation_plan.json` + los registros iniciales de clases CSS y comportamientos JS.

In [ ]:
# @title 🛠️ Generador de Metaprompt de Plan de Presentación
# @markdown Describe el contenido fuente (tema, resumen, o pega directamente el texto/notebook/paper) y su tipo de documento.

import json
import os
from string import Template
from ipywidgets import widgets
from IPython.display import display, HTML

contenido_fuente = input("Describe o pega el contenido fuente de la presentación: ")
tipo_documento = input("Tipo de documento (PDF / Jupyter Notebook / Markdown / Artículo científico / Documentación técnica / Código fuente / Resultados experimentales): ")

metaprompt_template_path = 'templetes/presentation_plan_meta_prompt.md'

if not os.path.exists(metaprompt_template_path):
    print(f"Error: La plantilla no se encontró en {metaprompt_template_path}")
    metaprompt_content = ""
else:
    with open(metaprompt_template_path, 'r', encoding='utf-8') as f:
        metaprompt_content = f.read()

# Reemplazo seguro para compatibilidad con string.Template (evita colisión con otras llaves {{...}} del propio prompt)
metaprompt_content = metaprompt_content.replace('{{contenido_fuente}}', '$contenido_fuente')
metaprompt_content = metaprompt_content.replace('{{tipo_documento}}', '$tipo_documento')
metaprompt_tpl = Template(metaprompt_content)

prompt_final = metaprompt_tpl.safe_substitute(
    contenido_fuente=contenido_fuente,
    tipo_documento=tipo_documento
)

print("\n✅ Prompt de plan procesado con éxito.\n")

prompt_json_esc = json.dumps(prompt_final)

boton_copiar_html = HTML(f"""
    <script>
    function copiarAlPortapapeles() {{
        const text = {prompt_json_esc};
        const textArea = document.createElement("textarea");
        textArea.value = text;
        document.body.appendChild(textArea);
        textArea.select();
        try {{
            document.execCommand('copy');
            alert("Prompt copiado al portapapeles");
        }} catch (err) {{
            console.error('Error al copiar: ', err);
        }}
        document.body.removeChild(textArea);
    }}
    </script>
    <button onclick="copiarAlPortapapeles()"
    style="background-color: #4CAF50; color: white; padding: 10px 20px; border: none; border-radius: 4px; cursor: pointer; margin-bottom: 10px;">
    📋 Copiar Prompt al Portapapeles
    </button>
""")

output_area = widgets.Textarea(
    value=prompt_final,
    layout=widgets.Layout(width='98%', height='300px'),
    description='Prompt:',
    disabled=False
)

display(boton_copiar_html)
display(output_area)

## 📝 2. Gestor de Plan de Presentación

Pega aquí la respuesta completa de la IA al prompt anterior (los tres bloques JSON: `presentation_plan.json`, `class_registry.json`, `js_registry.json`) y guárdala en `workflow/workflow/presentation_plan.md`. Este archivo es la única fuente de verdad que consultará el generador de prompts de sesión.

In [ ]:
# @title
import os
from ipywidgets import widgets
from IPython.display import display

file_path = 'workflow/workflow/presentation_plan.md'

text_area = widgets.Textarea(
    placeholder='Pegue aquí la respuesta completa de la IA (los 3 bloques JSON)...',
    description='Contenido:',
    layout=widgets.Layout(width='98%', height='400px')
)

save_button = widgets.Button(
    description='💾 Guardar Plan de Presentación',
    button_style='success',
    layout=widgets.Layout(width='220px')
)

clear_button = widgets.Button(
    description='🗑️ Limpiar Área',
    button_style='warning',
    layout=widgets.Layout(width='200px')
)

output = widgets.Output()

def on_save_clicked(b):
    with output:
        output.clear_output()
        try:
            os.makedirs(os.path.dirname(file_path), exist_ok=True)
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(text_area.value)
            print(f"✅ Archivo guardado exitosamente en: {file_path}")
            print(f"📏 Tamaño: {len(text_area.value)} caracteres.")
        except Exception as e:
            print(f"❌ Error al guardar: {str(e)}")

def on_clear_clicked(b):
    text_area.value = ''
    with output:
        output.clear_output()
        print("🧹 Área de texto limpiada.")

save_button.on_click(on_save_clicked)
clear_button.on_click(on_clear_clicked)

print("📝 Gestor de Plan de Presentación")
display(text_area)
display(widgets.HBox([save_button, clear_button]))
display(output)

## ⚙️ 3. Procesador del Plan (PRA)

Extrae los tres bloques JSON de `presentation_plan.md`, crea la carpeta del proyecto (usando `folder_name`), una subcarpeta por sesión, y guarda `class_registry.json` / `js_registry.json` — los registros vivos que se irán actualizando conforme construyas cada sesión. También genera un `manifest_draft.blade.php` con la estructura de secciones y comentarios guía, listo para ir pegando las entradas `<x-slide>` que produzca cada sesión.

In [ ]:
# @title
import json
import re
import os


def slugify(text):
    """Convierte el nombre del proyecto en un nombre de carpeta válido."""
    if not text:
        return "presentacion_nueva"
    text = text.lower().replace(" ", "_")
    return re.sub(r'(?u)[^-\w.]', '', text)


def extract_presentation_blocks(content):
    """
    Extrae los bloques JSON relevantes del archivo del plan.
    Retorna: (plan_principal, class_registry, js_registry)
    """
    json_blocks = re.findall(r'```json\s*(\{.*?\})\s*```', content, re.DOTALL)

    plan_principal = None
    class_registry = {"clases": []}
    js_registry = {"comportamientos": []}

    for block in json_blocks:
        try:
            data = json.loads(block)
        except json.JSONDecodeError:
            continue

        if "titulo" in data and "sesiones" in data:
            plan_principal = data
        elif "clases" in data:
            class_registry = data
        elif "comportamientos" in data:
            js_registry = data

    return plan_principal, class_registry, js_registry


def create_output_directory(input_file, project_name):
    input_dir = os.path.dirname(os.path.abspath(input_file))
    folder_name = slugify(project_name)
    output_path = os.path.join(input_dir, folder_name)

    if not os.path.exists(output_path):
        os.makedirs(output_path)
        print(f"Carpeta creada exitosamente en: {output_path}")
    else:
        print(f"Usando carpeta existente: {output_path}")

    return output_path


def get_output_path_from_plan(input_file, plan):
    project_name = plan.get('folder_name') or plan.get('titulo', 'presentacion_nueva')
    input_dir = os.path.dirname(os.path.abspath(input_file))
    return os.path.join(input_dir, slugify(project_name))


def save_registry(output_path, filename, data):
    path = os.path.join(output_path, filename)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    return path


def load_registry(output_path, filename, default):
    path = os.path.join(output_path, filename)
    if os.path.exists(path):
        try:
            with open(path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except json.JSONDecodeError:
            return default
    return default


def merge_registry_updates(existing, updates, list_key, name_key='nombre'):
    """Fusiona nuevas entradas evitando duplicados por nombre."""
    existing_names = {item.get(name_key) for item in existing.get(list_key, [])}
    for item in updates.get(list_key, []):
        if item.get(name_key) not in existing_names:
            existing.setdefault(list_key, []).append(item)
            existing_names.add(item.get(name_key))
    return existing


def build_manifest_draft(plan):
    lines = ["@extends('layouts.reveal')", "", "@section('title', $presentation->title)", "", "@section('slides')", ""]
    for sec in plan.get('sesiones', []):
        n = sec.get('nro', 0)
        titulo = sec.get('titulo_sesion', f'Sesión {n}')
        lines.append(f"    {{{{-- ========================================== --}}}}")
        lines.append(f"    {{{{-- SESIÓN {n}: {titulo.upper()} --}}}}")
        lines.append(f"    {{{{-- ========================================== --}}}}")
        lines.append(f'    <section data-title="{titulo}" data-session="sesion{n}">')
        for lam in sec.get('laminas', []):
            lines.append(f"        {{{{-- TODO: pegar aquí <x-slide view=\"sesion{n}.{lam.get('id')}\" data-title=\"{lam.get('data_title', '')}\" ... /> --}}}}")
        lines.append("    </section>")
        lines.append("")
    lines.append("@endsection")
    lines.append("")
    lines.append("@push('styles')")
    lines.append("    @include(\"external_slides::{$presentation->folder_name}.assets.styles\")")
    lines.append("@endpush")
    lines.append("")
    lines.append("@push('scripts')")
    lines.append("    @include(\"external_slides::{$presentation->folder_name}.assets.scripts\")")
    lines.append("@endpush")
    return "\n".join(lines)


def generate_presentation_project(input_file, project_name=None):
    if not os.path.exists(input_file):
        print(f"Error: No se encuentra {input_file}")
        return

    with open(input_file, 'r', encoding='utf-8') as f:
        content = f.read().replace('\u00a0', ' ')

    plan, class_registry, js_registry = extract_presentation_blocks(content)

    if not plan:
        print("Error: No se encontró el JSON del plan principal (debe contener 'titulo' y 'sesiones').")
        return

    if not project_name:
        project_name = plan.get('folder_name') or plan.get('titulo', 'presentacion_nueva')

    output_path = create_output_directory(input_file, project_name)

    # Subcarpetas por sesión
    for sec in plan.get('sesiones', []):
        n = sec.get('nro', 0)
        session_folder = os.path.join(output_path, f'sesion{n}')
        os.makedirs(session_folder, exist_ok=True)

    # Registros vivos
    save_registry(output_path, 'class_registry.json', class_registry)
    save_registry(output_path, 'js_registry.json', js_registry)

    # Borrador de manifest
    manifest_draft = build_manifest_draft(plan)
    with open(os.path.join(output_path, 'manifest_draft.blade.php'), 'w', encoding='utf-8') as f:
        f.write(manifest_draft)

    total_laminas = sum(len(sec.get('laminas', [])) for sec in plan.get('sesiones', []))

    print(f"✅ Proyecto inicializado en: {output_path}")
    print(f"📊 Sesiones: {len(plan.get('sesiones', []))} | Láminas planificadas: {total_laminas}")
    print(f"🎨 Clases CSS iniciales: {len(class_registry.get('clases', []))}")
    print(f"⚙️  Comportamientos JS iniciales: {len(js_registry.get('comportamientos', []))}")
    print("📄 manifest_draft.blade.php generado con la estructura de secciones y placeholders por lámina.")


if __name__ == "__main__":
    input_file = 'workflow/workflow/presentation_plan.md'

    if os.path.exists(input_file):
        print(f"🚀 Iniciando generación de proyecto para: {input_file}\n")
        generate_presentation_project(input_file)
    else:
        print(f"❌ Error: No se encontró {input_file}. Guarda primero el plan de presentación (paso 2).")

## 🎨 4. Construcción Asistida por Sesión

### 🚀 4.1 Generador de Prompt de Sesión

Selecciona el número de sesión y genera el prompt específico (plantilla `slide_group_meta_prompt.md`), inyectando automáticamente los objetivos/láminas de esa sesión desde el plan, y el estado **actual** de los registros de clases CSS y comportamientos JS (para que la IA no dupliqué nada ya definido en sesiones anteriores).

In [ ]:
# @title Generador de Prompts de Sesión
import os
import json
from ipywidgets import widgets
from IPython.display import display, HTML

input_plan_path = 'workflow/workflow/presentation_plan.md'
template_path = 'templetes/slide_group_meta_prompt.md'


def generate_session_prompt_on_the_fly(session_number):
    if not os.path.exists(input_plan_path):
        return f"❌ Error: No se encontró el archivo del plan en {input_plan_path}."

    if not os.path.exists(template_path):
        return f"❌ Error: No se encontró la plantilla en {template_path}."

    with open(input_plan_path, 'r', encoding='utf-8') as f:
        content = f.read().replace('\u00a0', ' ')

    plan, _, _ = extract_presentation_blocks(content)

    if not plan:
        return "❌ Error: No se pudo extraer el plan principal."

    sessions = plan.get('sesiones', [])
    sec = next((s for s in sessions if s.get('nro') == session_number), None)

    if not sec:
        return f"❌ Error: No se encontró la sesión {session_number} en el plan."

    output_path = get_output_path_from_plan(input_plan_path, plan)
    class_registry_actual = load_registry(output_path, 'class_registry.json', {"clases": []})
    js_registry_actual = load_registry(output_path, 'js_registry.json', {"comportamientos": []})

    replacements = {
        "{{session_number}}": str(session_number),
        "{{project_title}}": plan.get('titulo', 'Presentación'),
        "{{folder_name}}": plan.get('folder_name', slugify(plan.get('titulo', 'presentacion'))),
        "{{session_title}}": sec.get('titulo_sesion', f'Sesión {session_number}'),
        "{{objetivos}}": json.dumps(sec.get('objetivos', []), ensure_ascii=False),
        "{{laminas_json}}": json.dumps(sec.get('laminas', []), ensure_ascii=False, indent=2),
        "{{class_registry_actual}}": json.dumps(class_registry_actual, ensure_ascii=False, indent=2),
        "{{js_registry_actual}}": json.dumps(js_registry_actual, ensure_ascii=False, indent=2),
    }

    with open(template_path, 'r', encoding='utf-8') as f:
        prompt_template = f.read()

    for key, value in replacements.items():
        prompt_template = prompt_template.replace(key, value)

    return prompt_template


session_input = widgets.IntText(value=1, description='Sesión:', layout=widgets.Layout(width='150px'))
btn_generate = widgets.Button(description='⚡ Generar desde Plantilla', button_style='primary')
out_area = widgets.Output()


def on_gen_clicked(b):
    with out_area:
        out_area.clear_output()
        p_content = generate_session_prompt_on_the_fly(session_input.value)

        if p_content.startswith("❌"):
            print(p_content)
            return

        p_esc = json.dumps(p_content)
        copy_btn = HTML(f"""
            <script>
            function copyDynamicSession() {{
                const t = {p_esc};
                const el = document.createElement('textarea'); el.value = t;
                document.body.appendChild(el); el.select();
                document.execCommand('copy'); document.body.removeChild(el);
                alert('Prompt de sesión copiado');
            }}
            </script>
            <button onclick="copyDynamicSession()" style="background:#28a745;color:white;padding:8px;border:none;border-radius:4px;cursor:pointer;">📋 Copiar Prompt</button>
        """)

        display(copy_btn)
        display(widgets.Textarea(value=p_content, layout=widgets.Layout(width='98%', height='350px')))


btn_generate.on_click(on_gen_clicked)
display(widgets.HBox([session_input, btn_generate]), out_area)

### 🚀 4.2 Procesador Directo: IA → Archivos Finales

Pega aquí la respuesta completa de la IA para una sesión (los cinco bloques: láminas Blade, adición a `styles.blade.php`, adición a `scripts.blade.php`, entradas de `manifest.blade.php`, y actualización de registros). El procesador:

1. Extrae cada archivo Blade (identificado por su comentario `{{-- sesionN/id.blade.php --}}`) y lo guarda en su ruta real.
2. Anexa el bloque CSS a `styles.blade.php` (acumulado) y guarda también una copia aislada por sesión.
3. Anexa el bloque JS a `scripts.blade.php` (acumulado) y guarda también una copia aislada por sesión.
4. Guarda las entradas de `<x-slide>` en `manifest_additions/sesionN.blade.php`, listas para pegar en el manifest real.
5. Fusiona el bloque de registro (`nuevas_clases` / `nuevos_comportamientos`) dentro de `class_registry.json` / `js_registry.json`, **sin duplicar** nombres ya existentes — así la siguiente sesión parte del estado correcto.

In [ ]:
# @title
import os
import json
import re
from ipywidgets import widgets
from IPython.display import display


def process_session_content(content, session_number, output_path):
    results = []

    try:
        os.makedirs(output_path, exist_ok=True)

        # --- BLOQUE 1: archivos Blade de láminas ---
        blade_fences = re.findall(r'```blade\s*(.*?)```', content, re.DOTALL)
        combined_blade_text = '\n'.join(blade_fences)

        path_pattern = re.compile(r'\{\{--\s*(sesion\d+/[\w\-]+\.blade\.php)\s*--\}\}')
        matches = list(path_pattern.finditer(combined_blade_text))

        laminas_creadas = 0
        for i, m in enumerate(matches):
            rel_path = m.group(1)
            start = m.end()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(combined_blade_text)
            file_content = combined_blade_text[start:end].strip()

            full_path = os.path.join(output_path, rel_path)
            os.makedirs(os.path.dirname(full_path), exist_ok=True)
            with open(full_path, 'w', encoding='utf-8') as f:
                f.write(file_content + '\n')
            laminas_creadas += 1

        if laminas_creadas:
            results.append(f"✅ {laminas_creadas} archivo(s) Blade de lámina guardado(s).")
        else:
            results.append("⚠️ No se detectaron archivos Blade con el patrón {{-- sesionN/id.blade.php --}}.")

        # --- BLOQUE 2: adición de estilos ---
        css_match = re.search(r'```css\s*(.*?)```', content, re.DOTALL)
        if css_match:
            css_content = css_match.group(1).strip()

            session_css_dir = os.path.join(output_path, 'styles_additions')
            os.makedirs(session_css_dir, exist_ok=True)
            with open(os.path.join(session_css_dir, f'sesion{session_number}_styles.css'), 'w', encoding='utf-8') as f:
                f.write(css_content)

            master_styles_path = os.path.join(output_path, 'styles.blade.php')
            with open(master_styles_path, 'a', encoding='utf-8') as f:
                f.write('\n\n' + css_content + '\n')

            results.append("✅ CSS anexado a styles.blade.php (y guardado por separado en styles_additions/).")
        else:
            results.append("ℹ️ Sin bloque CSS nuevo en esta sesión.")

        # --- BLOQUE 3: adición de scripts ---
        js_match = re.search(r'```javascript\s*(.*?)```', content, re.DOTALL)
        if js_match:
            js_content = js_match.group(1).strip()

            session_js_dir = os.path.join(output_path, 'scripts_additions')
            os.makedirs(session_js_dir, exist_ok=True)
            with open(os.path.join(session_js_dir, f'sesion{session_number}_scripts.js'), 'w', encoding='utf-8') as f:
                f.write(js_content)

            master_scripts_path = os.path.join(output_path, 'scripts.blade.php')
            with open(master_scripts_path, 'a', encoding='utf-8') as f:
                f.write('\n\n' + js_content + '\n')

            results.append("✅ JS anexado a scripts.blade.php (y guardado por separado en scripts_additions/).")
        else:
            results.append("ℹ️ Sin bloque JS nuevo en esta sesión.")

        # --- BLOQUE 4: entradas de manifest ---
        # Toma el último bloque ```blade``` que contenga <x-slide> como el bloque de manifest,
        # o busca explícitamente un bloque separado si la IA lo devolvió aparte.
        manifest_candidates = [b for b in blade_fences if '<x-slide' in b]
        if manifest_candidates:
            manifest_content = manifest_candidates[-1].strip()
            manifest_dir = os.path.join(output_path, 'manifest_additions')
            os.makedirs(manifest_dir, exist_ok=True)
            with open(os.path.join(manifest_dir, f'sesion{session_number}.blade.php'), 'w', encoding='utf-8') as f:
                f.write(manifest_content)
            results.append(f"✅ Entradas de manifest guardadas en manifest_additions/sesion{session_number}.blade.php")
        else:
            results.append("⚠️ No se detectaron entradas <x-slide> para el manifest.")

        # --- BLOQUE 5: actualización de registros ---
        json_blocks = re.findall(r'```json\s*(\{.*?\})\s*```', content, re.DOTALL)
        registry_update = None
        for block in json_blocks:
            try:
                data = json.loads(block)
                if 'nuevas_clases' in data or 'nuevos_comportamientos' in data:
                    registry_update = data
                    break
            except json.JSONDecodeError:
                continue

        if registry_update:
            class_registry = load_registry(output_path, 'class_registry.json', {"clases": []})
            js_registry = load_registry(output_path, 'js_registry.json', {"comportamientos": []})

            class_registry = merge_registry_updates(
                class_registry,
                {"clases": registry_update.get('nuevas_clases', [])},
                'clases'
            )
            js_registry = merge_registry_updates(
                js_registry,
                {"comportamientos": registry_update.get('nuevos_comportamientos', [])},
                'comportamientos'
            )

            save_registry(output_path, 'class_registry.json', class_registry)
            save_registry(output_path, 'js_registry.json', js_registry)

            results.append(
                f"✅ Registros actualizados: +{len(registry_update.get('nuevas_clases', []))} clase(s), "
                f"+{len(registry_update.get('nuevos_comportamientos', []))} comportamiento(s)."
            )
        else:
            results.append("⚠️ No se detectó bloque de actualización de registros (BLOQUE 5).")

        return "\n".join(results)

    except Exception as e:
        return f"❌ Error crítico: {str(e)}"


def resolve_current_output_path():
    input_plan_path = 'workflow/workflow/presentation_plan.md'
    if not os.path.exists(input_plan_path):
        return None
    with open(input_plan_path, 'r', encoding='utf-8') as f:
        content = f.read().replace('\u00a0', ' ')
    plan, _, _ = extract_presentation_blocks(content)
    if not plan:
        return None
    return get_output_path_from_plan(input_plan_path, plan)


print("🚀 Procesador Directo de Sesión")
session_input = widgets.Text(value='1', description='Sesión #:', layout=widgets.Layout(width='200px'))
text_capture = widgets.Textarea(placeholder='Pegue aquí la respuesta completa de la IA para esta sesión...', description='Contenido:', layout=widgets.Layout(width='98%', height='350px'))
process_btn = widgets.Button(description='⚡ Procesar Sesión', button_style='success', layout=widgets.Layout(width='220px', margin='10px 0px'))
status_out = widgets.Output()


def on_button_clicked(b):
    with status_out:
        status_out.clear_output()
        if not text_capture.value.strip():
            return
        output_path = resolve_current_output_path()
        if not output_path:
            print("❌ No se pudo resolver la carpeta del proyecto. Corre primero el Procesador del Plan (paso 3).")
            return
        print("⏳ Procesando...")
        try:
            session_num = int(session_input.value)
        except ValueError:
            session_num = session_input.value
        print(process_session_content(text_capture.value, session_num, output_path))


process_btn.on_click(on_button_clicked)
display(session_input, text_capture, process_btn, status_out)

## 📦 5. Compresor y Descargador de Resultados

Comprime la carpeta de resultados del proyecto (copiando también `presentation_plan.md`) en un `.zip` y lo descarga a tu equipo local.

In [ ]:
# @title
import os
import shutil
from google.colab import files

input_plan_path = 'workflow/workflow/presentation_plan.md'

if not os.path.exists(input_plan_path):
    print(f"❌ Error: no se encontró {input_plan_path}.")
else:
    with open(input_plan_path, 'r', encoding='utf-8') as f:
        content = f.read().replace('\u00a0', ' ')
    plan, _, _ = extract_presentation_blocks(content)

    if not plan:
        print("❌ Error: no se pudo extraer el plan principal para localizar la carpeta del proyecto.")
    else:
        folder_to_zip = get_output_path_from_plan(input_plan_path, plan)
        zip_filename = 'outputs.zip'

        plan_destination = os.path.join(folder_to_zip, os.path.basename(input_plan_path))

        if os.path.exists(folder_to_zip):
            try:
                shutil.copy(input_plan_path, plan_destination)
                print(f"✅ Archivo '{input_plan_path}' copiado a '{plan_destination}'.")
            except Exception as e:
                print(f"❌ Error al copiar '{input_plan_path}': {e}")

            shutil.make_archive('outputs', 'zip', folder_to_zip)
            print(f"✅ Carpeta '{folder_to_zip}' comprimida exitosamente.")
            files.download('outputs.zip')
        else:
            print(f"❌ Error: no se encontró la carpeta '{folder_to_zip}'. Corre primero el Procesador del Plan (paso 3).")

## 🧹 Limpieza de Espacio de Trabajo

Elimina permanentemente `workflow/` y `outputs.zip`. Útil para reiniciar el proceso antes de una nueva presentación. Descomenta la línea para ejecutarlo.

In [ ]:
#rm -r workflow/ outputs.zip